# 06 · Paper: Cosmological Analysis Pipeline

Complete ML-based cosmological analysis for **ΛCDM** and **w₀wₐCDM** models.

| Section | Model | Dataset(s) |
|---------|-------|-----------|
| 1 | ΛCDM | panth + BAO |
| 2 | w₀wₐCDM | 3 comparative pairs (with/without CMB) |
| 3 | w₀wₐCDM | Final summary: BAO / BAO+panth / BAO+panth+CMB |

**Standard pipeline per scenario**: (1) Build dataset · (2) Train XGBoost · (3) SHAP · (4) MCMC · (5) GetDist

**Global parameter ranges** (dataset generation AND MCMC priors):
`Om ∈ [0.1, 0.9]`, `H0 ∈ [20, 100]`, `w0 ∈ [−3, 0.2]`, `wa ∈ [−3, 2]`

**CMB** = Planck 2018 Gaussian priors on Ωm and H₀ (from `cosmoml.priors.planck_prior_chi2`).


## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import getdist
import getdist.plots
from iminuit import Minuit

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from cosmoml.data import load_pantheon_plus, load_des_2024, load_des_2025, load_desi_bao
from cosmoml.theory import chi2_sne, chi2_sne_des, chi2_bao, chi2_joint
from cosmoml.priors import planck_prior_chi2
from cosmoml.sampling import build_chi2_dataset, load_or_build
from cosmoml.ml import (
    train_xgb, plot_learning_curve,
    shap_summary, shap_waterfall, shap_dependence_all,
    plot_corner_marginal, use_paper_style,
)
from cosmoml.config import OUTPUTS_DIR, PLANCK_H0, PLANCK_RD

use_paper_style()

NB_NAME    = "06_Paper"
DATASETS_DIR = OUTPUTS_DIR / "datasets"
MODELS_DIR   = OUTPUTS_DIR / "models"
FIGURES_DIR  = OUTPUTS_DIR / "figures" / NB_NAME
CHAINS_DIR   = OUTPUTS_DIR / "chains"  / NB_NAME
for _d in (DATASETS_DIR, MODELS_DIR, FIGURES_DIR, CHAINS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

FORCE_RETRAIN = False

panth   = load_pantheon_plus(apply_mask=True)
des2024 = load_des_2024()
des2025 = load_des_2025()
bao     = load_desi_bao()

print(f"Pantheon+SH0ES : {len(panth)} SNe ({panth.is_calib.sum()} calibrators)")
print(f"DES 2024       : {len(des2024)} SNe")
print(f"DES 2025       : {len(des2025)} SNe")
print(f"DESI BAO       : {len(bao)} measurements")


## Global Ranges, Labels & Helper Functions

In [ ]:
# ── Global parameter ranges (used for dataset generation AND MCMC priors) ──
GLOBAL_RANGES = {
    "Om": (0.1,   0.9),
    "H0": (20.0, 100.0),
    "w0": (-3.0,  0.2),
    "wa": (-3.0,  2.0),
}

LABELS = {
    "Om": r"$\Omega_m$",
    "H0": r"$H_0$",
    "w0": r"$w_0$",
    "wa": r"$w_a$",
}

MARKERS_LCDM = {"w0": -1.0, "wa": 0.0}   # ΛCDM reference lines for w0waCDM plots


# ── Helper: Minuit best-fit ───────────────────────────────────────────────────
def locate_bestfit(chi2_fn, features, ranges):
    """Auto-detect chi2 minimum with Minuit using global ranges as limits."""
    init = {f: (ranges[f][0] + ranges[f][1]) / 2.0 for f in features}
    # Better starting point for DE parameters
    if "w0" in features:
        init["w0"] = -1.0
    if "wa" in features:
        init["wa"] = 0.0
    if "Om" in features:
        init["Om"] = 0.3
    if "H0" in features:
        init["H0"] = 68.0
    m = Minuit(chi2_fn, **init)
    m.limits = [ranges[f] for f in features]
    m.migrad()
    REF = {f: float(m.values[f]) for f in features}
    print(f"  Best-fit : {', '.join(f'{f}={v:.4f}' for f, v in REF.items())}")
    print(f"  chi2_min : {m.fval:.2f}")
    return REF, m.fval


# ── Helper: build training dataset ───────────────────────────────────────────
def build_dataset(chi2_fn, section, features, ranges):
    """Minuit best-fit -> slices + random box -> cached CSV dataset."""
    csv_path = DATASETS_DIR / f"{section}_dataset.csv"

    REF, chi2_min = locate_bestfit(chi2_fn, features, ranges)
    ndim = len(features)

    def builder():
        if ndim == 2:
            slices = [
                {features[0]: ranges[features[0]],
                 features[1]: REF[features[1]], "_n": 5_000},
                {features[1]: ranges[features[1]],
                 features[0]: REF[features[0]], "_n": 5_000},
            ]
            n_random = 15_000
        else:
            # All 6 parameter pairs for 4D
            slices = []
            for _i in range(ndim):
                for _j in range(_i + 1, ndim):
                    fi, fj = features[_i], features[_j]
                    fixed  = {f: REF[f] for f in features if f not in (fi, fj)}
                    slices.append({fi: ranges[fi], fj: ranges[fj],
                                   **fixed, "_n": 6_000})
            n_random = 25_000

        return build_chi2_dataset(
            chi2_fn=chi2_fn,
            param_names=features,
            slices=slices,
            random_box={f: ranges[f] for f in features},
            n_random=n_random,
            save_to=csv_path,
            seed=42,
        )

    df = load_or_build(csv_path, builder, force=FORCE_RETRAIN)
    print(f"  Dataset  : {len(df):,} rows | chi2 range [{df['chi2'].min():.2f}, {df['chi2'].max():.2f}]")
    return df, REF


# ── Helper: train XGBoost + SHAP suite ──────────────────────────────────────
def train_and_shap(df, features, section, title=""):
    """Train XGBoost (log-chi2), learning curve, full SHAP suite."""
    model, info = train_xgb(
        df, features=features,
        log_target=True,
        hp_overrides=dict(n_estimators=3000, learning_rate=0.03, max_depth=10),
        cache_path=MODELS_DIR / f"{section}_model.ubj",
        force_retrain=FORCE_RETRAIN,
    )
    plot_learning_curve(
        info,
        title=f"{title} — Learning Curve  (R²={info['r2']:.5f})",
        show=True,
    )
    # SHAP
    shap_v, X_s = shap_summary(
        model, info["X_val"],
        title=f"{title} — SHAP (log₁₀ χ²)",
        show=True,
    )
    shap_waterfall(shap_v, idx=0,
                   title=f"{title} — SHAP waterfall (sample 0)",
                   show=True)
    shap_dependence_all(shap_v, X_s, show=True)
    return model, info, shap_v, X_s


# ── Helper: MCMC + single GetDist (cached) ───────────────────────────────────
def run_mcmc_and_getdist(model, features, ranges, ref, section,
                         labels, markers=None, title=""):
    """Run MCMC (or load cached .npy chain) and draw individual GetDist plot."""
    chain_path = CHAINS_DIR / f"{section}_samples.npy"
    str_labels = [labels.get(f, f).replace("$", "") for f in features]

    if chain_path.exists() and not FORCE_RETRAIN:
        samples = np.load(chain_path)
        print(f"  Loaded chain : {chain_path}  ({len(samples):,} samples)")
        mc = getdist.MCSamples(
            samples=samples, names=features, labels=str_labels,
            settings={"smooth_scale_2D": 0.5, "smooth_scale_1D": 0.5},
        )
        g = getdist.plots.get_subplot_plotter()
        g.triangle_plot(
            mc, filled=True, contour_colors=["#0044cc"],
            markers=markers,
            marker_args={"ls": "--", "color": "gray", "lw": 1.5, "alpha": 0.8}
            if markers else None,
        )
        if title:
            g.fig.suptitle(title, fontsize=12, y=1.01)
        plt.show()
    else:
        samples = plot_corner_marginal(
            model, features, ranges,
            labels=labels, ref=ref, markers=markers, title=title,
            n_chains=512, n_steps=5_000, burn_in=400, ess_target=5_000,
            smooth_scale=0.5, show=True,
        )
        np.save(chain_path, samples)
        print(f"  Saved chain  : {chain_path}")
    return samples


# ── Helper: comparative GetDist (multiple chains) ────────────────────────────
def plot_getdist_comparison(samples_list, dataset_labels, features, labels,
                             markers=None, title="", save_path=None):
    """Overlay multiple MCMC chains in one GetDist triangle plot."""
    COLORS = ["#0044cc", "#cc0000", "#009933", "#cc6600"]
    str_labels = [labels.get(f, f).replace("$", "") for f in features]

    mc_list = []
    for samps, dlabel in zip(samples_list, dataset_labels):
        mc = getdist.MCSamples(
            samples=samps,
            names=features,
            labels=str_labels,
            label=dlabel,
            settings={"smooth_scale_2D": 0.5, "smooth_scale_1D": 0.5},
        )
        mc_list.append(mc)

    g = getdist.plots.get_subplot_plotter()
    g.triangle_plot(
        mc_list,
        filled=True,
        contour_colors=COLORS[:len(mc_list)],
        markers=markers,
        marker_args={"ls": "--", "color": "gray", "lw": 1.5, "alpha": 0.8}
        if markers else None,
        legend_labels=dataset_labels,
        legend_loc="upper right",
    )
    if title:
        g.fig.suptitle(title, fontsize=13, y=1.01)
    if save_path:
        g.fig.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"  Saved figure : {save_path}")
    plt.show()
    return g.fig


# Section 1: Simple Scenario — ΛCDM (`panth + BAO`)

**Model**: `FlatLambdaCDM` · Parameters: `Om`, `H0`
**Data**: Pantheon+SH0ES (1657 SNe, Cepheid calibrators) + DESI BAO DR2 (13 measurements)
**Pipeline**: Build dataset → Train XGBoost → SHAP → MCMC → GetDist


In [ ]:
# ── Chi2 function (module-level for picklability with ProcessPoolExecutor) ──
def chi2_lcdm_panth_bao(Om, H0):
    """Joint ΛCDM chi2: Pantheon+SH0ES + DESI BAO."""
    return (
        chi2_sne(panth, "FlatLambdaCDM", Om=Om, H0=H0,
                 M="marginalize", use_cepheid_calibrators=True)
        + chi2_bao(bao, Om=Om, H0=H0)
    )

SECTION_1  = "6_1"
FEATURES_1 = ["Om", "H0"]
RANGES_1   = {k: GLOBAL_RANGES[k] for k in FEATURES_1}
TITLE_1    = "ΛCDM · Pantheon+SH0ES + DESI BAO"


In [ ]:
# Step 1: Build dataset
print(f"=== {TITLE_1} · Step 1: Build Dataset ===")
df_1, REF_1 = build_dataset(chi2_lcdm_panth_bao, SECTION_1, FEATURES_1, RANGES_1)


In [ ]:
# Step 2: Train XGBoost  |  Step 3: SHAP
print(f"=== {TITLE_1} · Steps 2-3: Train + SHAP ===")
model_1, info_1, shap_v_1, X_s_1 = train_and_shap(
    df_1, FEATURES_1, SECTION_1, title=TITLE_1,
)


In [ ]:
# Steps 4-5: MCMC + GetDist
print(f"=== {TITLE_1} · Steps 4-5: MCMC + GetDist ===")
samples_1 = run_mcmc_and_getdist(
    model_1, FEATURES_1, RANGES_1, REF_1, SECTION_1,
    labels=LABELS, markers=None, title=TITLE_1,
)


# Section 2: Complex Scenario — w₀wₐCDM (Comparative Pairs)

Each pair contrasts a dataset combination **without** and **with** CMB Planck priors,
showing how CMB tightens the dark-energy parameter contours.

| Pair | Without CMB | With CMB |
|------|-------------|----------|
| 1 | panth + BAO | panth + BAO + CMB |
| 2 | DES-2024 + BAO | DES-2024 + BAO + CMB |
| 3 | DES-2025 + BAO | DES-2025 + BAO + CMB |

**CMB** = Planck 2018 Gaussian priors on Ωm and H₀:
`chi2 += ((Om − 0.3153)/0.0073)² + ((H0 − 67.36)/0.54)²`


In [ ]:
# Shared w0waCDM setup
FEATURES_4  = ["Om", "H0", "w0", "wa"]
RANGES_4    = GLOBAL_RANGES.copy()
MARKERS_DE  = {"w0": -1.0, "wa": 0.0}


## Pair 1: `panth + BAO` vs `panth + BAO + CMB`


In [ ]:
# ── Pair 1 chi2 functions (module-level) ─────────────────────────────────────
def chi2_panth_bao(Om, H0, w0, wa):
    """w0waCDM: Pantheon+SH0ES + DESI BAO."""
    return chi2_joint(panth, bao, Om=Om, H0=H0, w0=w0, wa=wa)

def chi2_panth_bao_cmb(Om, H0, w0, wa):
    """w0waCDM: Pantheon+SH0ES + DESI BAO + Planck CMB priors."""
    return chi2_panth_bao(Om, H0, w0, wa) + planck_prior_chi2(Om=Om, H0=H0)


In [ ]:
# ── Pair 1a: panth + BAO ─────────────────────────────────────────────────────
SECTION_2_1A = "6_2_1a"
TITLE_2_1A   = "w₀wₐCDM · Pantheon+SH0ES + DESI BAO"

print(f"=== {TITLE_2_1A} · Steps 1-3 ===")
df_2_1a, REF_2_1a = build_dataset(chi2_panth_bao, SECTION_2_1A, FEATURES_4, RANGES_4)
model_2_1a, info_2_1a, shap_v_2_1a, X_s_2_1a = train_and_shap(
    df_2_1a, FEATURES_4, SECTION_2_1A, title=TITLE_2_1A,
)


In [ ]:
# Pair 1a — Steps 4-5: MCMC + GetDist
print(f"=== {TITLE_2_1A} · Steps 4-5 ===")
samples_2_1a = run_mcmc_and_getdist(
    model_2_1a, FEATURES_4, RANGES_4, REF_2_1a, SECTION_2_1A,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_2_1A,
)


In [ ]:
# ── Pair 1b: panth + BAO + CMB ───────────────────────────────────────────────
SECTION_2_1B = "6_2_1b"
TITLE_2_1B   = "w₀wₐCDM · Pantheon+SH0ES + DESI BAO + CMB"

print(f"=== {TITLE_2_1B} · Steps 1-3 ===")
df_2_1b, REF_2_1b = build_dataset(chi2_panth_bao_cmb, SECTION_2_1B, FEATURES_4, RANGES_4)
model_2_1b, info_2_1b, shap_v_2_1b, X_s_2_1b = train_and_shap(
    df_2_1b, FEATURES_4, SECTION_2_1B, title=TITLE_2_1B,
)


In [ ]:
# Pair 1b — Steps 4-5: MCMC + GetDist
print(f"=== {TITLE_2_1B} · Steps 4-5 ===")
samples_2_1b = run_mcmc_and_getdist(
    model_2_1b, FEATURES_4, RANGES_4, REF_2_1b, SECTION_2_1B,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_2_1B,
)


In [ ]:
# ── Pair 1: Comparative GetDist (panth+BAO vs panth+BAO+CMB) ─────────────────
plot_getdist_comparison(
    [samples_2_1a, samples_2_1b],
    ["Pantheon+ + BAO", "Pantheon+ + BAO + CMB"],
    FEATURES_4, LABELS,
    markers=MARKERS_DE,
    title="w₀wₐCDM — Pair 1: Pantheon+SH0ES + BAO  vs  + CMB",
    save_path=FIGURES_DIR / "6_2_pair1_comparison.png",
)


## Pair 2: `DES-2024 + BAO` vs `DES-2024 + BAO + CMB`


In [ ]:
# ── Pair 2 chi2 functions (module-level) ─────────────────────────────────────
def chi2_des2024_bao(Om, H0, w0, wa):
    """w0waCDM: DES-SN5YR 2024 + DESI BAO."""
    return (
        chi2_sne_des(des2024, "Flatw0waCDM", Om=Om, H0=H0, w0=w0, wa=wa)
        + chi2_bao(bao, Om=Om, H0=H0, w0=w0, wa=wa)
    )

def chi2_des2024_bao_cmb(Om, H0, w0, wa):
    """w0waCDM: DES-SN5YR 2024 + DESI BAO + Planck CMB priors."""
    return chi2_des2024_bao(Om, H0, w0, wa) + planck_prior_chi2(Om=Om, H0=H0)


In [ ]:
# ── Pair 2a: DES-2024 + BAO ───────────────────────────────────────────────────
SECTION_2_2A = "6_2_2a"
TITLE_2_2A   = "w₀wₐCDM · DES-SN5YR 2024 + DESI BAO"

print(f"=== {TITLE_2_2A} · Steps 1-3 ===")
df_2_2a, REF_2_2a = build_dataset(chi2_des2024_bao, SECTION_2_2A, FEATURES_4, RANGES_4)
model_2_2a, info_2_2a, shap_v_2_2a, X_s_2_2a = train_and_shap(
    df_2_2a, FEATURES_4, SECTION_2_2A, title=TITLE_2_2A,
)


In [ ]:
# Pair 2a — Steps 4-5: MCMC + GetDist
print(f"=== {TITLE_2_2A} · Steps 4-5 ===")
samples_2_2a = run_mcmc_and_getdist(
    model_2_2a, FEATURES_4, RANGES_4, REF_2_2a, SECTION_2_2A,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_2_2A,
)


In [ ]:
# ── Pair 2b: DES-2024 + BAO + CMB ────────────────────────────────────────────
SECTION_2_2B = "6_2_2b"
TITLE_2_2B   = "w₀wₐCDM · DES-SN5YR 2024 + DESI BAO + CMB"

print(f"=== {TITLE_2_2B} · Steps 1-3 ===")
df_2_2b, REF_2_2b = build_dataset(chi2_des2024_bao_cmb, SECTION_2_2B, FEATURES_4, RANGES_4)
model_2_2b, info_2_2b, shap_v_2_2b, X_s_2_2b = train_and_shap(
    df_2_2b, FEATURES_4, SECTION_2_2B, title=TITLE_2_2B,
)


In [ ]:
# Pair 2b — Steps 4-5: MCMC + GetDist
print(f"=== {TITLE_2_2B} · Steps 4-5 ===")
samples_2_2b = run_mcmc_and_getdist(
    model_2_2b, FEATURES_4, RANGES_4, REF_2_2b, SECTION_2_2B,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_2_2B,
)


In [ ]:
# ── Pair 2: Comparative GetDist (DES-2024+BAO vs DES-2024+BAO+CMB) ───────────
plot_getdist_comparison(
    [samples_2_2a, samples_2_2b],
    ["DES-2024 + BAO", "DES-2024 + BAO + CMB"],
    FEATURES_4, LABELS,
    markers=MARKERS_DE,
    title="w₀wₐCDM — Pair 2: DES-SN5YR 2024 + BAO  vs  + CMB",
    save_path=FIGURES_DIR / "6_2_pair2_comparison.png",
)


## Pair 3: `DES-2025 + BAO` vs `DES-2025 + BAO + CMB`


In [ ]:
# ── Pair 3 chi2 functions (module-level) ─────────────────────────────────────
def chi2_des2025_bao(Om, H0, w0, wa):
    """w0waCDM: DES-SN5YR 2025 + DESI BAO."""
    return (
        chi2_sne_des(des2025, "Flatw0waCDM", Om=Om, H0=H0, w0=w0, wa=wa)
        + chi2_bao(bao, Om=Om, H0=H0, w0=w0, wa=wa)
    )

def chi2_des2025_bao_cmb(Om, H0, w0, wa):
    """w0waCDM: DES-SN5YR 2025 + DESI BAO + Planck CMB priors."""
    return chi2_des2025_bao(Om, H0, w0, wa) + planck_prior_chi2(Om=Om, H0=H0)


In [ ]:
# ── Pair 3a: DES-2025 + BAO ───────────────────────────────────────────────────
SECTION_2_3A = "6_2_3a"
TITLE_2_3A   = "w₀wₐCDM · DES-SN5YR 2025 + DESI BAO"

print(f"=== {TITLE_2_3A} · Steps 1-3 ===")
df_2_3a, REF_2_3a = build_dataset(chi2_des2025_bao, SECTION_2_3A, FEATURES_4, RANGES_4)
model_2_3a, info_2_3a, shap_v_2_3a, X_s_2_3a = train_and_shap(
    df_2_3a, FEATURES_4, SECTION_2_3A, title=TITLE_2_3A,
)


In [ ]:
# Pair 3a — Steps 4-5: MCMC + GetDist
print(f"=== {TITLE_2_3A} · Steps 4-5 ===")
samples_2_3a = run_mcmc_and_getdist(
    model_2_3a, FEATURES_4, RANGES_4, REF_2_3a, SECTION_2_3A,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_2_3A,
)


In [ ]:
# ── Pair 3b: DES-2025 + BAO + CMB ────────────────────────────────────────────
SECTION_2_3B = "6_2_3b"
TITLE_2_3B   = "w₀wₐCDM · DES-SN5YR 2025 + DESI BAO + CMB"

print(f"=== {TITLE_2_3B} · Steps 1-3 ===")
df_2_3b, REF_2_3b = build_dataset(chi2_des2025_bao_cmb, SECTION_2_3B, FEATURES_4, RANGES_4)
model_2_3b, info_2_3b, shap_v_2_3b, X_s_2_3b = train_and_shap(
    df_2_3b, FEATURES_4, SECTION_2_3B, title=TITLE_2_3B,
)


In [ ]:
# Pair 3b — Steps 4-5: MCMC + GetDist
print(f"=== {TITLE_2_3B} · Steps 4-5 ===")
samples_2_3b = run_mcmc_and_getdist(
    model_2_3b, FEATURES_4, RANGES_4, REF_2_3b, SECTION_2_3B,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_2_3B,
)


In [ ]:
# ── Pair 3: Comparative GetDist (DES-2025+BAO vs DES-2025+BAO+CMB) ───────────
plot_getdist_comparison(
    [samples_2_3a, samples_2_3b],
    ["DES-2025 + BAO", "DES-2025 + BAO + CMB"],
    FEATURES_4, LABELS,
    markers=MARKERS_DE,
    title="w₀wₐCDM — Pair 3: DES-SN5YR 2025 + BAO  vs  + CMB",
    save_path=FIGURES_DIR / "6_2_pair3_comparison.png",
)


# Section 3: Final Summary — w₀wₐCDM

Single GetDist plot comparing three dataset combinations to show the
progressive constraining power of each dataset:

1. **BAO** — DESI BAO DR2 only
2. **BAO + Pantheon+** — reused from Section 2, Pair 1a
3. **BAO + Pantheon+ + CMB** — reused from Section 2, Pair 1b


In [ ]:
# ── BAO-only chi2 (module-level) ─────────────────────────────────────────────
def chi2_bao_only(Om, H0, w0, wa):
    """w0waCDM: DESI BAO only."""
    return chi2_bao(bao, Om=Om, H0=H0, w0=w0, wa=wa)

SECTION_3_BAO = "6_3_bao"
TITLE_3_BAO   = "w₀wₐCDM · DESI BAO only"


In [ ]:
# ── Section 3: BAO-only pipeline (Steps 1-5) ─────────────────────────────────
print(f"=== {TITLE_3_BAO} · Steps 1-3 ===")
df_3_bao, REF_3_bao = build_dataset(chi2_bao_only, SECTION_3_BAO, FEATURES_4, RANGES_4)
model_3_bao, info_3_bao, shap_v_3_bao, X_s_3_bao = train_and_shap(
    df_3_bao, FEATURES_4, SECTION_3_BAO, title=TITLE_3_BAO,
)


In [ ]:
# BAO-only — Steps 4-5: MCMC + GetDist
print(f"=== {TITLE_3_BAO} · Steps 4-5 ===")
samples_3_bao = run_mcmc_and_getdist(
    model_3_bao, FEATURES_4, RANGES_4, REF_3_bao, SECTION_3_BAO,
    labels=LABELS, markers=MARKERS_DE, title=TITLE_3_BAO,
)


In [ ]:
# ── Section 3: Final 3-way summary GetDist ───────────────────────────────────
# Reuse chains from Section 2 Pair 1 for BAO+panth and BAO+panth+CMB
_chain_panthbao = CHAINS_DIR / "6_2_1a_samples.npy"
_chain_full     = CHAINS_DIR / "6_2_1b_samples.npy"

if not _chain_panthbao.exists() or not _chain_full.exists():
    raise FileNotFoundError(
        "Section 2 Pair-1 chains not found. Run Section 2 Pair 1 first."
    )

samples_summary_bao      = samples_3_bao
samples_summary_panthbao = np.load(_chain_panthbao)
samples_summary_full     = np.load(_chain_full)

print(f"BAO only     : {len(samples_summary_bao):,} samples")
print(f"BAO+panth    : {len(samples_summary_panthbao):,} samples")
print(f"BAO+panth+CMB: {len(samples_summary_full):,} samples")

plot_getdist_comparison(
    [samples_summary_bao, samples_summary_panthbao, samples_summary_full],
    ["BAO", "BAO + Pantheon+", "BAO + Pantheon+ + CMB"],
    FEATURES_4, LABELS,
    markers=MARKERS_DE,
    title="w₀wₐCDM — Summary: BAO  /  BAO+Pantheon+  /  BAO+Pantheon++CMB",
    save_path=FIGURES_DIR / "6_3_summary_getdist.png",
)
